# Tutorial 4, Q1b: Icki-Toffee, weighted objective on normalised data

*Original script: `5 Source Files/Python/icki_toffee_weighted_norm.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

In [ ]:
# Define integer variables
model.x = Var(range(12), domain=NonNegativeIntegers)

In [ ]:
# Define objective function coefficients
cost = [12, 12, 12, 15, 13, 13, 16, 16, 15, 10, 18, 20]
pop = [10, 10, 8, 9, 5, 6, 7, 8, 10, 3, 8, 4]
fat = [20, 16, 10, 7, 6, 5, 13, 15, 18, 0, 19, 0]
cals = [100, 100, 80, 50, 60, 40, 80, 90, 80, 145, 120, 120]
weight = [24, 22, 15, 12, 10, 10, 14, 17, 20, 8, 20, 15]

In [ ]:
# choose alpha between 0 and 1
alpha = 0.59

In [ ]:
# Objective function
model.obj = Objective(
    expr=(1 - alpha) * sum(pop[i] * model.x[i] / max(pop) for i in model.x) - alpha * sum(cost[i] * model.x[i] / max(cost) for i in model.x),
    sense=maximize
)

In [ ]:
# Define the inequality constraints
model.weight_min = Constraint(expr=sum(weight[i] * model.x[i] for i in model.x) >= 320)
model.weight_max = Constraint(expr=sum(weight[i] * model.x[i] for i in model.x) <= 1.5 * 320)
model.cals_max = Constraint(expr=sum(cals[i] * model.x[i] for i in model.x) <= 2850)
model.fat_weight_min = Constraint(expr=sum(fat[i] * model.x[i] for i in model.x) <= 0.45 * sum(weight[i] * model.x[i] for i in model.x))

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')  # You can use a different solver here
solver.solve(model, tee=True)

In [ ]:
# Print the model
model.pprint()

In [ ]:
# Print the results
model.display()